In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy import stats

def createdata():
  data = {
      'Age': np.random.randint(18, 70, size=20),
      'Salary': np.random.randint(30000, 120000, size=20),
      'Purchased': np.random.choice([0, 1], size=20),
      'Gender': np.random.choice(['Male', 'Female'], size=20),
      'City': np.random.choice(['New York', 'San Francisco', 'Los Angeles'], size=20)
  }

  df = pd.DataFrame(data)
  return df

df = createdata()
df.head(10)

df.shape

# Introduce some missing values for demonstration
df.loc[5, 'Age'] = np.nan
df.loc[10, 'Salary'] = np.nan
df.head(10)

# Basic information about the dataset
print(df.info())

# Summary statistics
print(df.describe())

#Code to Find Missing Values
# Check for missing values in each column
missing_values = df.isnull().sum()

# Display columns with missing values
print(missing_values[missing_values > 0])

#Set the values to some value (zero, the mean, the median, etc.).
# Step 1: Create an instance of SimpleImputer with the median strategy for Age and mean stratergy for Salary
imputer1 = SimpleImputer(strategy="median")
imputer2 = SimpleImputer(strategy="mean")

df_copy=df

# Step 2: Fit the imputer on the "Age" and "Salary"column
# Note: SimpleImputer expects a 2D array, so we reshape the column
imputer1.fit(df_copy[["Age"]])
imputer2.fit(df_copy[["Salary"]])

# Step 3: Transform (fill) the missing values in the "Age" and "Salary"c column
df_copy["Age"] = imputer1.transform(df[["Age"]])
df_copy["Salary"] = imputer2.transform(df[["Salary"]])

# Verify that there are no missing values left
print(df_copy["Age"].isnull().sum())
print(df_copy["Salary"].isnull().sum())

#Handling Categorical Attributes
#Using Ordinal Encoding for gender COlumn and One-Hot Encoding for City Column

# Initialize OrdinalEncoder
ordinal_encoder = OrdinalEncoder(categories=[["Male", "Female"]])
# Fit and transform the data
df_copy["Gender_Encoded"] = ordinal_encoder.fit_transform(df_copy[["Gender"]])

# Initialize OneHotEncoder
onehot_encoder = OneHotEncoder()

# Fit and transform the "City" column
encoded_data = onehot_encoder.fit_transform(df[["City"]])

# Convert the sparse matrix to a dense array
encoded_array = encoded_data.toarray()

# Convert to DataFrame for better visualization
encoded_df = pd.DataFrame(encoded_array, columns=onehot_encoder.get_feature_names_out(["City"]))
df_encoded = pd.concat([df_copy, encoded_df], axis=1)

df_encoded.drop("Gender", axis=1, inplace=True)
df_encoded.drop("City", axis=1, inplace=True)

print(df_encoded. head())

#Data Transformation
# Min-Max Scaler/Normalization (range 0-1)
#Pros: Keeps all data between 0 and 1; ideal for distance-based models.
#Cons: Can distort data distribution, especially with extreme outliers.
normalizer = MinMaxScaler()
df_encoded[['Salary']] = normalizer.fit_transform(df_encoded[['Salary']])
df_encoded.head()

# Standardization (mean=0, variance=1)
#Pros: Works well for normally distributed data; suitable for many models.
#Cons: Sensitive to outliers.
scaler = StandardScaler()
df_encoded[['Age']] = scaler.fit_transform(df_encoded[['Age']])
df_encoded.head()

#Removing Outliers
# Outlier Detection and Treatment using IQR
#Pros: Simple and effective for mild outliers.
#Cons: May overly reduce variation if there are many extreme outliers.
df_encoded_copy1=df_encoded
df_encoded_copy2=df_encoded
df_encoded_copy3=df_encoded

Q1 = df_encoded_copy1['Salary'].quantile(0.25)
Q3 = df_encoded_copy1['Salary'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df_encoded_copy1['Salary'] = np.where(df_encoded_copy1['Salary'] > upper_bound, upper_bound,
                        np.where(df_encoded_copy1['Salary'] < lower_bound, lower_bound, df_encoded_copy1['Salary']))

print(df_encoded_copy1.head())

#Removing Outliers
# Z-score method
#Pros: Good for normally distributed data.
#Cons: Not suitable for non-normal data; may miss outliers in skewed distributions.

df_encoded_copy2['Salary_zscore'] = stats.zscore(df_encoded_copy2['Salary'])
df_encoded_copy2['Salary'] = np.where(df_encoded_copy2['Salary_zscore'].abs() > 3, np.nan, df_encoded_copy2['Salary'])  # Replace outliers with NaN
print(df_encoded_copy2.head())

#Removing Outliers
# Median replacement for outliers
#Pros: Keeps distribution shape intact, useful when capping isn’t feasible.
#Cons: May distort data if outliers represent real phenomena.
df_encoded_copy3['Salary_zscore'] = stats.zscore(df_encoded_copy3['Salary'])
median_salary = df_encoded_copy3['Salary'].median()
df_encoded_copy3['Salary'] = np.where(df_encoded_copy3['Salary_zscore'].abs() > 3, median_salary, df_encoded_copy3['Salary'])
print(df_encoded_copy3.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Age        19 non-null     float64
 1   Salary     19 non-null     float64
 2   Purchased  20 non-null     int64  
 3   Gender     20 non-null     object 
 4   City       20 non-null     object 
dtypes: float64(2), int64(1), object(2)
memory usage: 932.0+ bytes
None
             Age         Salary  Purchased
count  19.000000      19.000000  20.000000
mean   38.894737   72608.578947   0.450000
std    16.309148   28443.906183   0.510418
min    18.000000   30584.000000   0.000000
25%    24.000000   48738.000000   0.000000
50%    35.000000   64649.000000   0.000000
75%    52.000000   96763.000000   1.000000
max    66.000000  117338.000000   1.000000
Age       1
Salary    1
dtype: int64
0
0
    Age    Salary  Purchased  Gender_Encoded  City_Los Angeles  City_New York  \
0  30.0   42707.0          0         

In [4]:
from google.colab import files
uploaded = files.upload()


Saving adult.csv to adult (1).csv


In [5]:
from google.colab import files
uploaded = files.upload()


Saving Diabetes .csv to Diabetes  (1).csv


In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler


In [11]:
diabetes_df = pd.read_csv("Diabetes .csv")
adult_df = pd.read_csv("adult.csv")

print("Datasets loaded successfully")


Datasets loaded successfully


In [10]:
import os
os.listdir()


['.config', 'Diabetes .csv', '.ipynb_checkpoints', 'adult.csv', 'sample_data']

In [12]:
print("\nDiabetes Dataset Info")
diabetes_df.info()

print("\nAdult Income Dataset Info")
adult_df.info()



Diabetes Dataset Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   ID         1000 non-null   int64  
 1   No_Pation  1000 non-null   int64  
 2   Gender     1000 non-null   object 
 3   AGE        1000 non-null   int64  
 4   Urea       1000 non-null   float64
 5   Cr         1000 non-null   int64  
 6   HbA1c      1000 non-null   float64
 7   Chol       1000 non-null   float64
 8   TG         1000 non-null   float64
 9   HDL        1000 non-null   float64
 10  LDL        1000 non-null   float64
 11  VLDL       1000 non-null   float64
 12  BMI        1000 non-null   float64
 13  CLASS      1000 non-null   object 
dtypes: float64(8), int64(4), object(2)
memory usage: 109.5+ KB

Adult Income Dataset Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null 

In [13]:
print("\nDiabetes Dataset Statistics")
print(diabetes_df.describe())

print("\nAdult Income Dataset Statistics")
print(adult_df.describe())



Diabetes Dataset Statistics
                ID     No_Pation          AGE         Urea           Cr  \
count  1000.000000  1.000000e+03  1000.000000  1000.000000  1000.000000   
mean    340.500000  2.705514e+05    53.528000     5.124743    68.943000   
std     240.397673  3.380758e+06     8.799241     2.935165    59.984747   
min       1.000000  1.230000e+02    20.000000     0.500000     6.000000   
25%     125.750000  2.406375e+04    51.000000     3.700000    48.000000   
50%     300.500000  3.439550e+04    55.000000     4.600000    60.000000   
75%     550.250000  4.538425e+04    59.000000     5.700000    73.000000   
max     800.000000  7.543566e+07    79.000000    38.900000   800.000000   

             HbA1c         Chol           TG          HDL          LDL  \
count  1000.000000  1000.000000  1000.000000  1000.000000  1000.000000   
mean      8.281160     4.862820     2.349610     1.204750     2.609790   
std       2.534003     1.301738     1.401176     0.660414     1.115102   

In [14]:
if "Ocean Proximity" in adult_df.columns:
    print("\nOcean Proximity Value Counts:")
    print(adult_df["Ocean Proximity"].value_counts())
else:
    print("\nOcean Proximity column not present in dataset")



Ocean Proximity column not present in dataset


In [15]:
print("\nMissing Values in Diabetes Dataset")
print(diabetes_df.isnull().sum()[diabetes_df.isnull().sum() > 0])

print("\nMissing Values in Adult Income Dataset")
print(adult_df.isnull().sum()[adult_df.isnull().sum() > 0])



Missing Values in Diabetes Dataset
Series([], dtype: int64)

Missing Values in Adult Income Dataset
Series([], dtype: int64)


In [18]:
# Diabetes dataset – numerical columns
for col in diabetes_df.select_dtypes(include=np.number):
    diabetes_df[col] = diabetes_df[col].fillna(diabetes_df[col].mean())

# Adult dataset – categorical columns
for col in adult_df.select_dtypes(include="object"):
    adult_df[col] = adult_df[col].fillna(adult_df[col].mode()[0])



In [19]:
adult_df = pd.get_dummies(adult_df, drop_first=True)


In [21]:
def remove_outliers(df):
    numeric_df = df.select_dtypes(include=np.number)

    Q1 = numeric_df.quantile(0.25)
    Q3 = numeric_df.quantile(0.75)
    IQR = Q3 - Q1

    condition = ~((numeric_df < (Q1 - 1.5 * IQR)) |
                  (numeric_df > (Q3 + 1.5 * IQR))).any(axis=1)

    return df[condition]


In [23]:
num_cols_diabetes = diabetes_df.select_dtypes(include=np.number).columns


In [24]:
minmax = MinMaxScaler()

diabetes_scaled = diabetes_df.copy()
diabetes_scaled[num_cols_diabetes] = minmax.fit_transform(diabetes_df[num_cols_diabetes])

print("Min-Max Scaled Diabetes Dataset")
print(diabetes_scaled.head())


Min-Max Scaled Diabetes Dataset
         ID  No_Pation Gender       AGE      Urea        Cr     HbA1c  \
0  0.627034   0.000237      F  0.508475  0.109375  0.050378  0.264901   
1  0.918648   0.000452      M  0.101695  0.104167  0.070529  0.264901   
2  0.524406   0.000634      F  0.508475  0.109375  0.050378  0.264901   
3  0.849812   0.001160      F  0.508475  0.109375  0.050378  0.264901   
4  0.629537   0.000452      M  0.220339  0.171875  0.050378  0.264901   

       Chol        TG       HDL       LDL      VLDL       BMI CLASS  
0  0.407767  0.044444  0.226804  0.114583  0.011461  0.173913     N  
1  0.359223  0.081481  0.092784  0.187500  0.014327  0.139130     N  
2  0.407767  0.044444  0.226804  0.114583  0.011461  0.173913     N  
3  0.407767  0.044444  0.226804  0.114583  0.011461  0.173913     N  
4  0.475728  0.051852  0.061856  0.177083  0.008596  0.069565     N  


In [25]:
standard = StandardScaler()

adult_scaled = adult_df.copy()
adult_scaled[adult_df.columns] = standard.fit_transform(adult_df)

print("\nStandard Scaled Adult Income Dataset")
print(adult_scaled.head())



Standard Scaled Adult Income Dataset
        age    fnlwgt  educational-num  capital-gain  capital-loss  \
0 -0.995129  0.351675        -1.197259     -0.144804     -0.217127   
1 -0.046942 -0.945524        -0.419335     -0.144804     -0.217127   
2 -0.776316  1.394723         0.747550     -0.144804     -0.217127   
3  0.390683 -0.277844        -0.030373      0.886874     -0.217127   
4 -1.505691 -0.815954        -0.030373     -0.144804     -0.217127   

   hours-per-week  workclass_Federal-gov  workclass_Local-gov  \
0       -0.034087              -0.173795            -0.261940   
1        0.772930              -0.173795            -0.261940   
2       -0.034087              -0.173795             3.817672   
3       -0.034087              -0.173795            -0.261940   
4       -0.841104              -0.173795            -0.261940   

   workclass_Never-worked  workclass_Private  ...  native-country_Puerto-Rico  \
0                -0.01431           0.663711  ...                   -